# CallWhisper-8k: Kaggle Whisper-small LoRA Pilot

Purpose: run a credible compact-model adaptation experiment for Indian telephone-style Hindi ASR.

This notebook trains a LoRA adapter on `openai/whisper-small`, then evaluates both the base HF model and the adapted model on the same frozen GramVaani manifests. It is designed to answer one honest question:

> On this fixed slice, did a compact LoRA-adapted Whisper-small model change WER/CER versus base HF Whisper-small?

Do not report a final fine-tuned-model claim unless the run uses `GV_Train_100h`, excludes frozen benchmark IDs, and saves the base-vs-LoRA comparison results.


## Inputs And Outputs

Kaggle inputs can be either attached folders or attached tarballs. The notebook can also download both labelled GramVaani archives from OpenSLR when internet is enabled.

Supported attached dataset layout:

```text
/kaggle/input/<dataset-name>/
  GV_Train_100h/
    Audio/*
    text
    mp3.scp or wav.scp
  GV_Dev_5h/
    Audio/*
    text
    mp3.scp or wav.scp
```

Supported attached archive layout:

```text
/kaggle/input/<dataset-name>/GV_Train_100h.tar.gz
/kaggle/input/<dataset-name>/GV_Dev_5h.tar.gz
```

Main outputs:

```text
/kaggle/working/saved_datasets/        # reusable downloaded/copied tarballs
/kaggle/working/checkpoints/           # LoRA adapter checkpoints
/kaggle/working/results/               # split CSVs, config JSON, metrics JSON/Markdown
```


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/anshulLuhsna/CallWhisper-8k.git"
REPO_DIR = Path("/kaggle/working/CallWhisper-8k")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
TEMP_ROOT = Path("/kaggle/temp") if Path("/kaggle").exists() else WORKING_DIR / "temp"
LOCAL_DATA_ROOT = TEMP_ROOT / "data"
SAVED_DATASETS_DIR = WORKING_DIR / "saved_datasets"
RESULTS_DIR = WORKING_DIR / "results"
ARTIFACTS_DIR = RESULTS_DIR / "whisper_small_lora_pilot"
CHECKPOINT_ROOT = WORKING_DIR / "checkpoints"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = str(REPO_DIR / "src")
for directory in [TEMP_ROOT, LOCAL_DATA_ROOT, SAVED_DATASETS_DIR, RESULTS_DIR, ARTIFACTS_DIR, CHECKPOINT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

print("Repo:", Path.cwd())
print("Kaggle input root:", KAGGLE_INPUT_ROOT)
print("Temp extracted data root:", LOCAL_DATA_ROOT)
print("Saved datasets dir:", SAVED_DATASETS_DIR)
print("Results dir:", ARTIFACTS_DIR)
print("Checkpoint root:", CHECKPOINT_ROOT)


## Run Configuration

Default is `pilot`. Use `smoke` only to test the pipeline quickly. Use `serious` only after one pilot run succeeds and the output metrics look sane.


In [ ]:
import json
import random
from datetime import datetime, timezone

RUN_PROFILE = "pilot"  # options: smoke, pilot, serious
SEED = 0
BASE_MODEL = "openai/whisper-small"
MIN_DURATION_S = 1.0
MAX_DURATION_S = 30.0
DECODE_BEAMS = [1, 5]
DOWNLOAD_AND_SAVE_GRAMVAANI = True

RUN_PROFILES = {
    "smoke": {
        "max_train_samples": 200,
        "max_internal_eval_samples": 25,
        "max_steps": 40,
        "requires_train_split": False,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 4,
        "gradient_accumulation_steps": 2,
        "learning_rate": 1e-4,
        "warmup_steps": 10,
        "eval_steps": 20,
        "save_steps": 20,
    },
    "pilot": {
        "max_train_samples": 3200,
        "max_internal_eval_samples": 200,
        "max_steps": 800,
        "requires_train_split": True,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 4,
        "gradient_accumulation_steps": 2,
        "learning_rate": 1e-4,
        "warmup_steps": 80,
        "eval_steps": 100,
        "save_steps": 100,
    },
    "serious": {
        "max_train_samples": 12000,
        "max_internal_eval_samples": 500,
        "max_steps": 2500,
        "requires_train_split": True,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 2,
        "gradient_accumulation_steps": 2,
        "learning_rate": 1e-4,
        "warmup_steps": 250,
        "eval_steps": 250,
        "save_steps": 250,
    },
}

if RUN_PROFILE not in RUN_PROFILES:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}; choose one of {sorted(RUN_PROFILES)}")

PROFILE = RUN_PROFILES[RUN_PROFILE]
RUN_NAME = f"whisper-small-lora-gramvaani-{RUN_PROFILE}-seed{SEED}"
CHECKPOINT_DIR = CHECKPOINT_ROOT / RUN_NAME
FINAL_ADAPTER_DIR = CHECKPOINT_DIR / "final_adapter"
PROCESSOR_DIR = CHECKPOINT_DIR / "processor"

LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "v_proj"],
    "lora_dropout": 0.05,
    "bias": "none",
}

random.seed(SEED)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    "run_profile": RUN_PROFILE,
    "run_name": RUN_NAME,
    "base_model": BASE_MODEL,
    "profile": PROFILE,
    "duration_filter": [MIN_DURATION_S, MAX_DURATION_S],
    "decode_beams": DECODE_BEAMS,
    "lora": LORA_CONFIG,
}, indent=2))


In [ ]:
!nvidia-smi
!python -m pip install -q -U pip
!python -m pip install -q --no-deps -e "."
!python -m pip install -q   "transformers==4.46.3"   "peft==0.13.2"   "accelerate==0.34.2"   "datasets==3.1.0"   "evaluate==0.4.3"   "jiwer>=3.0.4"   "librosa>=0.10"   "soundfile>=0.12"   "tensorboard>=2.16"   "pandas>=2.0"   "tabulate>=0.9"   "pyyaml>=6.0"   "tqdm>=4.66"
!which ffmpeg || echo "ffmpeg not found; MP3 duration/audio decode may fail without ffmpeg."
!which ffprobe || echo "ffprobe not found; duration filtering may fail without ffprobe."


In [ ]:
import importlib.metadata as importlib_metadata
import platform

PACKAGE_NAMES = [
    "torch",
    "transformers",
    "peft",
    "accelerate",
    "datasets",
    "evaluate",
    "jiwer",
    "librosa",
    "soundfile",
    "pandas",
    "tabulate",
    "pyyaml",
]

package_versions = {"python": platform.python_version(), "platform": platform.platform()}
for package_name in PACKAGE_NAMES:
    try:
        package_versions[package_name] = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        package_versions[package_name] = None

versions_path = ARTIFACTS_DIR / f"{RUN_NAME}_package_versions.json"
versions_path.write_text(json.dumps(package_versions, indent=2), encoding="utf-8")
print(json.dumps(package_versions, indent=2))
print("Wrote", versions_path)


In [ ]:
print("Available Kaggle inputs:")
if KAGGLE_INPUT_ROOT.exists():
    for input_path in sorted(KAGGLE_INPUT_ROOT.iterdir()):
        print("-", input_path)
else:
    print("/kaggle/input does not exist. Are you running this outside Kaggle?")


## Acquire GramVaani Data

The notebook saves reusable tarballs under `/kaggle/working/saved_datasets` and extracts temporary working copies under `/kaggle/temp/data`. Tar extraction is path-checked before extraction.


In [ ]:
import os
import shutil
import tarfile
import urllib.request
from collections import Counter
from pathlib import Path

GV_DATASETS = [
    {
        "name": "GV_Dev_5h",
        "url": "https://www.openslr.org/resources/118/GV_Dev_5h.tar.gz",
        "archive_name": "GV_Dev_5h.tar.gz",
        "min_size_bytes": 50_000_000,
    },
    {
        "name": "GV_Train_100h",
        "url": "https://www.openslr.org/resources/118/GV_Train_100h.tar.gz",
        "archive_name": "GV_Train_100h.tar.gz",
        "min_size_bytes": 1_000_000_000,
    },
]


def iter_search_roots() -> list[Path]:
    roots = []
    if KAGGLE_INPUT_ROOT.exists():
        roots.extend(sorted(KAGGLE_INPUT_ROOT.iterdir()))
    if LOCAL_DATA_ROOT.exists():
        roots.append(LOCAL_DATA_ROOT)
        roots.extend(sorted(LOCAL_DATA_ROOT.iterdir()))
    return roots


def find_attached_archive(archive_name: str) -> Path | None:
    if not KAGGLE_INPUT_ROOT.exists():
        return None
    matches = sorted(KAGGLE_INPUT_ROOT.rglob(archive_name))
    return matches[0] if matches else None


def find_dataset_dir(name: str) -> Path | None:
    for root in iter_search_roots():
        if root.name == name and root.is_dir():
            return root
        candidate = root / name
        if candidate.is_dir():
            return candidate
    return None


def safe_extract_tar(archive: Path, target_dir: Path) -> None:
    target_root = target_dir.resolve()
    with tarfile.open(archive, "r:gz") as tar:
        for member in tar.getmembers():
            if member.issym() or member.islnk():
                raise RuntimeError(f"Refusing to extract linked tar member: {member.name}")
            member_path = (target_dir / member.name).resolve()
            if member_path != target_root and target_root not in member_path.parents:
                raise RuntimeError(f"Unsafe tar path: {member.name}")
        tar.extractall(target_dir)


def ensure_archive(dataset: dict) -> Path:
    archive = SAVED_DATASETS_DIR / dataset["archive_name"]
    if archive.exists() and archive.stat().st_size >= dataset["min_size_bytes"]:
        print("Archive already saved:", archive)
        return archive

    attached_archive = find_attached_archive(dataset["archive_name"])
    if attached_archive is not None:
        print("Copying attached archive:", attached_archive, "->", archive)
        shutil.copy2(attached_archive, archive)
        if archive.stat().st_size < dataset["min_size_bytes"]:
            raise RuntimeError(f"Attached archive looks too small: {archive}")
        return archive

    if not DOWNLOAD_AND_SAVE_GRAMVAANI:
        existing_dir = find_dataset_dir(dataset["name"])
        if existing_dir is not None:
            print("Using attached extracted dataset without archive:", existing_dir)
            return archive
        raise FileNotFoundError(f"Missing {dataset['archive_name']} and download disabled")

    print("Downloading", dataset["url"])
    print("Saving archive to", archive)
    urllib.request.urlretrieve(dataset["url"], archive)
    if archive.stat().st_size < dataset["min_size_bytes"]:
        raise RuntimeError(f"Downloaded archive looks partial/corrupt: {archive}")
    print("Saved", archive, "size_gb=", round(archive.stat().st_size / 1_000_000_000, 3))
    return archive


def ensure_extracted(dataset: dict, archive: Path) -> Path:
    existing_dir = find_dataset_dir(dataset["name"])
    if existing_dir is not None:
        print("Dataset already available:", existing_dir)
        return existing_dir

    if not archive.exists():
        raise FileNotFoundError(f"No archive available to extract {dataset['name']}: {archive}")

    print("Extracting", archive, "to", LOCAL_DATA_ROOT)
    safe_extract_tar(archive, LOCAL_DATA_ROOT)
    extracted = find_dataset_dir(dataset["name"])
    if extracted is None:
        matches = sorted(str(p) for p in LOCAL_DATA_ROOT.glob(dataset["name"] + "*"))
        raise FileNotFoundError(f"Expected extracted folder {dataset['name']}; found {matches}")
    print("Extracted", extracted)
    return extracted


def validate_dataset_dir(dataset_dir: Path) -> dict:
    audio_dir = dataset_dir / "Audio"
    text_path = dataset_dir / "text"
    scp_path = dataset_dir / "mp3.scp"
    scp_type = "mp3"
    if not scp_path.exists():
        scp_path = dataset_dir / "wav.scp"
        scp_type = "wav"
    missing = [str(p) for p in [audio_dir, text_path, scp_path] if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Dataset {dataset_dir} is missing required files: {missing}")
    audio_count = sum(1 for p in audio_dir.iterdir() if p.is_file())
    if audio_count == 0:
        raise RuntimeError(f"No audio files found in {audio_dir}")
    return {
        "dataset_dir": str(dataset_dir),
        "audio_dir": str(audio_dir),
        "text_path": str(text_path),
        "scp_path": str(scp_path),
        "scp_type": scp_type,
        "audio_file_count": audio_count,
    }

available_datasets = {}
saved_dataset_records = []
for dataset in GV_DATASETS:
    archive = ensure_archive(dataset)
    dataset_dir = ensure_extracted(dataset, archive)
    validation = validate_dataset_dir(dataset_dir)
    available_datasets[dataset["name"]] = Path(validation["dataset_dir"])
    saved_dataset_records.append({
        "name": dataset["name"],
        "url": dataset["url"],
        "archive": str(archive) if archive.exists() else None,
        "archive_size_bytes": archive.stat().st_size if archive.exists() else None,
        **validation,
    })

index_path = SAVED_DATASETS_DIR / "gramvaani_saved_datasets.json"
index_path.write_text(json.dumps({
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source": "OpenSLR SLR118",
    "license_note": "Free for academic use; commercial use requires permission from Gram Vaani.",
    "datasets": saved_dataset_records,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote", index_path)
print(json.dumps(saved_dataset_records, indent=2))


## Link Data Into The Cloned Repo

The repo manifests refer to `datasets/GV_Dev_5h/...`, so this cell symlinks the acquired folders into the clone.


In [ ]:
import shutil

DATASETS_DIR = REPO_DIR / "datasets"
MANIFESTS_DIR = DATASETS_DIR / "manifests"


def link_dataset(name: str, source: Path) -> Path:
    target = DATASETS_DIR / name
    if target.exists() or target.is_symlink():
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(source, target)
    return target

for dataset_name, dataset_dir in available_datasets.items():
    linked_path = link_dataset(dataset_name, dataset_dir)
    print(dataset_name, "->", linked_path, "source=", dataset_dir)

# Optional: copied manifests from Kaggle input override repo manifests only when explicitly attached.
if KAGGLE_INPUT_ROOT.exists():
    manifest_dirs = sorted(KAGGLE_INPUT_ROOT.rglob("manifests"))
    for manifest_dir in manifest_dirs[:1]:
        if manifest_dir.is_dir():
            MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
            for src in manifest_dir.glob("*.csv"):
                dst = MANIFESTS_DIR / src.name
                shutil.copy2(src, dst)
                print("Copied attached manifest:", src, "->", dst)

GV_TRAIN = DATASETS_DIR / "GV_Train_100h"
GV_DEV = DATASETS_DIR / "GV_Dev_5h"
if PROFILE["requires_train_split"] and not GV_TRAIN.exists():
    raise FileNotFoundError(f"RUN_PROFILE={RUN_PROFILE} requires GV_Train_100h")

TRAIN_DATASET_DIR = GV_TRAIN if GV_TRAIN.exists() else GV_DEV
SMOKE_TEST_ONLY = TRAIN_DATASET_DIR.name != "GV_Train_100h"
if SMOKE_TEST_ONLY and RUN_PROFILE != "smoke":
    raise RuntimeError("Only RUN_PROFILE='smoke' may train from GV_Dev_5h. Pilot/serious require GV_Train_100h.")

print("Train dataset dir:", TRAIN_DATASET_DIR)
print("Smoke test only:", SMOKE_TEST_ONLY)


## Build Deterministic Train/Internal-Eval Splits

Frozen benchmark IDs are excluded. Clips outside `1-30s` are filtered to avoid Whisper's 30-second training window truncating audio while keeping full transcripts.


In [ ]:
import csv
import subprocess
from statistics import mean, median
from tqdm.auto import tqdm

FROZEN_MANIFESTS = [
    MANIFESTS_DIR / "gramvaani_dev_50.csv",
    MANIFESTS_DIR / "gramvaani_dev_50_8khz.csv",
    MANIFESTS_DIR / "gramvaani_dev_50_highrate.csv",
]

# Duration probing is the slow part because it shells out to ffprobe.
# Keep this False for Kaggle pilot/serious runs so the notebook stops as soon as it has enough valid rows.
PROBE_ALL_DURATIONS = RUN_PROFILE == "smoke"
DURATION_PROBE_BUFFER = 400 if RUN_PROFILE == "pilot" else 1000


def read_text(path: Path) -> dict[str, str]:
    rows = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            utt, _, text = line.partition(" ")
            text = text.strip()
            if utt and text:
                rows[utt] = text
    return rows


def read_scp(path: Path, dataset_dir: Path) -> dict[str, Path]:
    rows = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(maxsplit=1)
            if len(parts) != 2:
                continue
            utt, audio_text = parts
            if audio_text.endswith("|"):
                continue
            audio_path = Path(audio_text)
            if not audio_path.is_absolute():
                audio_path = dataset_dir / audio_path
            rows[utt] = audio_path
    return rows


def frozen_ids_from_manifest(path: Path) -> set[str]:
    ids = set()
    if not path.exists():
        raise FileNotFoundError(f"Missing frozen manifest: {path}")
    with path.open(encoding="utf-8") as f:
        for row in csv.DictReader(f):
            audio_path = row.get("audio_path", "")
            if audio_path:
                ids.add(Path(audio_path).stem)
    return ids


def duration_seconds(audio_path: Path) -> float:
    cmd = [
        "ffprobe",
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(audio_path),
    ]
    out = subprocess.check_output(cmd, text=True).strip()
    return float(out)


def write_rows_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = ["utt_id", "audio", "text", "duration_s"]
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, "") for key in fieldnames})


def write_exclusions_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = ["utt_id", "reason", "audio", "text", "duration_s", "detail"]
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, "") for key in fieldnames})


def load_unprobed_rows(dataset_dir: Path) -> tuple[list[dict], list[dict]]:
    text_path = dataset_dir / "text"
    scp_path = dataset_dir / "mp3.scp"
    if not scp_path.exists():
        scp_path = dataset_dir / "wav.scp"
    texts = read_text(text_path)
    audio_paths = read_scp(scp_path, dataset_dir)
    frozen_ids = set().union(*(frozen_ids_from_manifest(p) for p in FROZEN_MANIFESTS))

    rows = []
    exclusions = []
    for utt_id, text in tqdm(texts.items(), total=len(texts), desc="Indexing transcript/SCP rows"):
        audio_path = audio_paths.get(utt_id)
        if utt_id in frozen_ids:
            exclusions.append({"utt_id": utt_id, "reason": "frozen_benchmark", "text": text})
            continue
        if audio_path is None:
            exclusions.append({"utt_id": utt_id, "reason": "missing_scp_entry", "text": text})
            continue
        if not audio_path.exists():
            exclusions.append({"utt_id": utt_id, "reason": "missing_audio", "audio": str(audio_path), "text": text})
            continue
        rows.append({"utt_id": utt_id, "audio": str(audio_path), "text": text})
    return rows, exclusions


def filter_rows_by_duration(unprobed_rows: list[dict], base_exclusions: list[dict]) -> tuple[list[dict], list[dict], dict]:
    needed_rows = PROFILE["max_train_samples"] + PROFILE["max_internal_eval_samples"]
    target_valid_rows = needed_rows if PROBE_ALL_DURATIONS else needed_rows + DURATION_PROBE_BUFFER
    rows = []
    exclusions = list(base_exclusions)
    probe_failures = 0
    probed_rows = 0

    progress = tqdm(unprobed_rows, total=len(unprobed_rows), desc="Probing durations with ffprobe")
    for row in progress:
        probed_rows += 1
        audio_path = Path(row["audio"])
        try:
            duration_s = duration_seconds(audio_path)
        except Exception as exc:
            probe_failures += 1
            exclusions.append({**row, "reason": "duration_probe_failed", "detail": repr(exc)})
            continue

        rounded_duration = round(duration_s, 3)
        if duration_s < MIN_DURATION_S:
            exclusions.append({**row, "reason": "duration_too_short", "duration_s": rounded_duration})
            continue
        if duration_s > MAX_DURATION_S:
            exclusions.append({**row, "reason": "duration_too_long", "duration_s": rounded_duration})
            continue

        rows.append({**row, "duration_s": rounded_duration})
        progress.set_postfix(valid=len(rows), probed=probed_rows, failures=probe_failures)

        if not PROBE_ALL_DURATIONS and len(rows) >= target_valid_rows:
            break

    unprobed_after_quota = max(0, len(unprobed_rows) - probed_rows)
    duration_probe_summary = {
        "probe_all_durations": PROBE_ALL_DURATIONS,
        "needed_rows": needed_rows,
        "target_valid_rows": target_valid_rows,
        "probed_rows": probed_rows,
        "valid_rows_after_duration_filter": len(rows),
        "unprobed_after_quota": unprobed_after_quota,
        "duration_probe_failures": probe_failures,
    }
    return rows, exclusions, duration_probe_summary

unprobed_rows, base_excluded_rows = load_unprobed_rows(TRAIN_DATASET_DIR)
random.Random(SEED).shuffle(unprobed_rows)
candidate_rows, excluded_rows, duration_probe_summary = filter_rows_by_duration(unprobed_rows, base_excluded_rows)
needed_rows = PROFILE["max_train_samples"] + PROFILE["max_internal_eval_samples"]
if len(candidate_rows) < needed_rows:
    raise RuntimeError(
        f"Too few usable rows after duration filtering: {len(candidate_rows)}; needed {needed_rows}. "
        "Try RUN_PROFILE='smoke' or set PROBE_ALL_DURATIONS=True."
    )

train_rows = candidate_rows[: PROFILE["max_train_samples"]]
internal_eval_rows = candidate_rows[PROFILE["max_train_samples"] : PROFILE["max_train_samples"] + PROFILE["max_internal_eval_samples"]]

train_split_path = ARTIFACTS_DIR / f"{RUN_NAME}_train_split.csv"
internal_eval_split_path = ARTIFACTS_DIR / f"{RUN_NAME}_internal_eval_split.csv"
exclusions_path = ARTIFACTS_DIR / f"{RUN_NAME}_excluded_rows.csv"
write_rows_csv(train_split_path, train_rows)
write_rows_csv(internal_eval_split_path, internal_eval_rows)
write_exclusions_csv(exclusions_path, excluded_rows)

filtered_counts = Counter(row["reason"] for row in excluded_rows)
durations = [row["duration_s"] for row in train_rows + internal_eval_rows]
split_summary = {
    "candidate_rows_after_filter": len(candidate_rows),
    "train_rows": len(train_rows),
    "internal_eval_rows": len(internal_eval_rows),
    "excluded_rows": len(excluded_rows),
    "excluded_by_reason": dict(filtered_counts),
    "duration_probe": duration_probe_summary,
    "duration_summary": {
        "min": min(durations) if durations else None,
        "median": median(durations) if durations else None,
        "mean": mean(durations) if durations else None,
        "max": max(durations) if durations else None,
    },
    "train_split_path": str(train_split_path),
    "internal_eval_split_path": str(internal_eval_split_path),
    "exclusions_path": str(exclusions_path),
}
print(json.dumps(split_summary, ensure_ascii=False, indent=2))


In [ ]:
run_config = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_name": RUN_NAME,
    "run_profile": RUN_PROFILE,
    "seed": SEED,
    "base_model": BASE_MODEL,
    "profile": PROFILE,
    "lora": LORA_CONFIG,
    "duration_filter": {"min_s": MIN_DURATION_S, "max_s": MAX_DURATION_S},
    "decode_beams": DECODE_BEAMS,
    "train_dataset_dir": str(TRAIN_DATASET_DIR),
    "smoke_test_only": SMOKE_TEST_ONLY,
    "frozen_manifests": [str(p) for p in FROZEN_MANIFESTS],
    "split_summary": split_summary,
    "package_versions_path": str(versions_path),
    "saved_dataset_index": str(index_path),
}
run_config_path = ARTIFACTS_DIR / f"{RUN_NAME}_run_config.json"
run_config_path.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote", run_config_path)


## Prepare Whisper-small + LoRA

This adapts only attention projection weights (`q_proj`, `v_proj`). The goal is a compact edge-oriented model, not a claim that it should beat Hindi-tuned medium/large models.


In [ ]:
import numpy as np
import torch
from datasets import Audio, Dataset
from peft import LoraConfig, get_peft_model
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="Hindi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

lora_config = LoraConfig(**LORA_CONFIG)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_ds = Dataset.from_list(train_rows).cast_column("audio", Audio(sampling_rate=16000))
internal_eval_ds = Dataset.from_list(internal_eval_rows).cast_column("audio", Audio(sampling_rate=16000))
print(train_ds)
print(internal_eval_ds)


In [ ]:
def decode_audio_for_datasets(audio):
    # datasets 3.x can return either a dict or a torchcodec AudioDecoder-like object.
    if isinstance(audio, dict):
        return audio["array"], audio["sampling_rate"]
    if hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        array = samples.data
        if hasattr(array, "detach"):
            array = array.detach().cpu().numpy()
        array = np.asarray(array).squeeze()
        return array, int(samples.sample_rate)
    raise TypeError(f"Unsupported audio object: {type(audio)}")


def prepare_example(batch):
    array, sampling_rate = decode_audio_for_datasets(batch["audio"])
    batch["input_features"] = processor.feature_extractor(
        array,
        sampling_rate=sampling_rate,
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

train_ds = train_ds.map(prepare_example, remove_columns=train_ds.column_names)
internal_eval_ds = internal_eval_ds.map(prepare_example, remove_columns=internal_eval_ds.column_names)
print(train_ds)
print(internal_eval_ds)


## Train Adapter

The trainer monitors `eval_loss` on the internal non-frozen split and reloads the best checkpoint at the end. Frozen benchmark WER/CER is evaluated only after training.


In [ ]:
import inspect
from dataclasses import dataclass
from typing import Any
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if labels.shape[1] > 0 and (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

args_kwargs = dict(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=PROFILE["per_device_train_batch_size"],
    per_device_eval_batch_size=PROFILE["per_device_eval_batch_size"],
    gradient_accumulation_steps=PROFILE["gradient_accumulation_steps"],
    learning_rate=PROFILE["learning_rate"],
    warmup_steps=PROFILE["warmup_steps"],
    max_steps=PROFILE["max_steps"],
    weight_decay=0.01,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    logging_steps=25 if RUN_PROFILE != "smoke" else 5,
    eval_steps=PROFILE["eval_steps"],
    save_steps=PROFILE["save_steps"],
    save_total_limit=2,
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=False,
    report_to=["tensorboard"],
    remove_unused_columns=False,
    label_names=["labels"],
)

sig = inspect.signature(Seq2SeqTrainingArguments)
if "eval_strategy" in sig.parameters:
    args_kwargs["eval_strategy"] = "steps"
else:
    args_kwargs["evaluation_strategy"] = "steps"

training_args = Seq2SeqTrainingArguments(**args_kwargs)
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=internal_eval_ds,
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
    tokenizer=processor.feature_extractor,
)

train_result = trainer.train()
trainer.save_model(str(FINAL_ADAPTER_DIR))
processor.save_pretrained(str(PROCESSOR_DIR))
trainer_state_path = ARTIFACTS_DIR / f"{RUN_NAME}_trainer_state.json"
trainer.state.save_to_json(str(trainer_state_path))

train_metrics = dict(train_result.metrics)
train_metrics_path = ARTIFACTS_DIR / f"{RUN_NAME}_train_metrics.json"
train_metrics_path.write_text(json.dumps(train_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved adapter:", FINAL_ADAPTER_DIR)
print("Saved processor:", PROCESSOR_DIR)
print("Saved trainer state:", trainer_state_path)
print(json.dumps(train_metrics, indent=2))


## Evaluate Base vs LoRA On Frozen Manifests

This uses the same Hugging Face generation path for base Whisper-small and the LoRA-adapted model. It reports both macro mean per-file metrics and corpus-level metrics.


In [ ]:
import csv
import gc
import json
import re
import sys
import unicodedata
from pathlib import Path

import librosa
import pandas as pd
import torch
from jiwer import cer as jiwer_cer
from jiwer import wer as jiwer_wer
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import WhisperForConditionalGeneration

# Make this cell robust if Kaggle loses the editable install but the cloned repo still exists.
repo_src = str(REPO_DIR / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)

try:
    from callwhisper.eval.cer import character_error_rate
    from callwhisper.eval.normalizer import normalize_text
    from callwhisper.eval.wer import word_error_rate
except ModuleNotFoundError:
    _ASCII_PUNCTUATION = re.compile(r"[!\"#$%&'()*+,\-./:;<=>?@\[\\\]^_`{|}~]")
    _WHITESPACE = re.compile(r"\s+")

    def normalize_text(text: str) -> str:
        normalized = unicodedata.normalize("NFC", text).strip().lower()
        normalized = _ASCII_PUNCTUATION.sub(" ", normalized)
        normalized = _WHITESPACE.sub(" ", normalized)
        return normalized.strip()

    def word_error_rate(reference: str, hypothesis: str) -> float:
        return float(jiwer_wer(normalize_text(reference), normalize_text(hypothesis)))

    def character_error_rate(reference: str, hypothesis: str) -> float:
        return float(jiwer_cer(normalize_text(reference), normalize_text(hypothesis)))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EVAL_LIMIT = None  # Set to 10 for quick debugging only.


def find_optional_manifest(filename: str) -> Path | None:
    candidates = []
    repo_candidate = MANIFESTS_DIR / filename
    if repo_candidate.exists():
        candidates.append(repo_candidate)
    if KAGGLE_INPUT_ROOT.exists():
        candidates.extend(sorted(KAGGLE_INPUT_ROOT.rglob(filename)))
    return candidates[0] if candidates else None


def build_eval_manifests() -> list[Path]:
    manifests = list(FROZEN_MANIFESTS)
    fleurs_manifest = find_optional_manifest("fleurs_hi_clean_50.csv")
    if fleurs_manifest is not None:
        manifests.append(fleurs_manifest)
        print("Found optional FLEURS manifest:", fleurs_manifest)
    return manifests


def resolve_manifest_audio(path_value: str, manifest_path: Path) -> Path:
    p = Path(path_value)
    if p.is_absolute() and p.exists():
        return p
    candidates = [REPO_DIR / p, manifest_path.parent / p]
    if KAGGLE_INPUT_ROOT.exists():
        candidates.extend(root / p for root in KAGGLE_INPUT_ROOT.iterdir() if root.is_dir())
    for candidate in candidates:
        if candidate.exists():
            return candidate
    basename_matches = []
    if KAGGLE_INPUT_ROOT.exists():
        basename_matches.extend(sorted(KAGGLE_INPUT_ROOT.rglob(p.name)))
    basename_matches.extend(sorted(LOCAL_DATA_ROOT.rglob(p.name)))
    if basename_matches:
        return basename_matches[0]
    return REPO_DIR / p


def load_manifest_rows(manifest_path: Path) -> list[dict]:
    rows = list(csv.DictReader(manifest_path.open(encoding="utf-8")))
    if EVAL_LIMIT is not None:
        rows = rows[:EVAL_LIMIT]
    resolved = []
    for row in rows:
        audio_path = resolve_manifest_audio(row["audio_path"], manifest_path)
        if not audio_path.exists():
            raise FileNotFoundError(f"Eval audio missing: {audio_path} from {manifest_path}")
        resolved.append({**row, "resolved_audio_path": str(audio_path)})
    return resolved


def configure_generation_model(eval_model):
    eval_model.config.forced_decoder_ids = None
    eval_model.config.suppress_tokens = []
    eval_model.eval()
    return eval_model


def load_base_eval_model():
    eval_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL).to(DEVICE)
    return configure_generation_model(eval_model)


def load_lora_eval_model():
    base = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL).to(DEVICE)
    adapted = PeftModel.from_pretrained(base, FINAL_ADAPTER_DIR)
    adapted = adapted.merge_and_unload().to(DEVICE)
    return configure_generation_model(adapted)


def transcribe_one(eval_model, audio_path: Path, num_beams: int) -> str:
    array, _ = librosa.load(str(audio_path), sr=16000, mono=True)
    inputs = processor.feature_extractor(array, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(DEVICE)
    with torch.no_grad():
        pred_ids = eval_model.generate(
            input_features,
            language="hi",
            task="transcribe",
            num_beams=num_beams,
            do_sample=False,
            max_new_tokens=225,
        )
    return processor.batch_decode(pred_ids, skip_special_tokens=True)[0].strip()


def summarize_predictions(predictions: list[dict]) -> dict:
    if not predictions:
        raise ValueError("Cannot summarize empty predictions")
    refs = [item["reference_text"] for item in predictions]
    hyps = [item["hypothesis_text"] for item in predictions]
    refs_norm = [normalize_text(text) for text in refs]
    hyps_norm = [normalize_text(text) for text in hyps]
    return {
        "files": len(predictions),
        "macro_wer": float(sum(item["wer"] for item in predictions) / len(predictions)),
        "macro_cer": float(sum(item["cer"] for item in predictions) / len(predictions)),
        "corpus_wer": float(jiwer_wer(refs_norm, hyps_norm)),
        "corpus_cer": float(jiwer_cer(refs_norm, hyps_norm)),
    }


def evaluate_model_on_manifest(eval_model, model_label: str, manifest_path: Path, num_beams: int) -> dict:
    rows = load_manifest_rows(manifest_path)
    predictions = []
    for row in tqdm(rows, desc=f"{model_label} {manifest_path.stem} beam{num_beams}"):
        audio_path = Path(row["resolved_audio_path"])
        hypothesis = transcribe_one(eval_model, audio_path, num_beams=num_beams)
        reference = row["reference_text"]
        predictions.append({
            "model": model_label,
            "base_model": BASE_MODEL,
            "run_name": RUN_NAME,
            "manifest": str(manifest_path),
            "slice": row.get("slice", manifest_path.stem),
            "condition": row.get("condition", "unknown"),
            "language": row.get("language", "hi"),
            "num_beams": num_beams,
            "audio_path": str(audio_path),
            "reference_text": reference,
            "hypothesis_text": hypothesis,
            "wer": word_error_rate(reference, hypothesis),
            "cer": character_error_rate(reference, hypothesis),
        })
    summary = summarize_predictions(predictions)
    payload = {
        "summary": {
            **summary,
            "model": model_label,
            "base_model": BASE_MODEL,
            "run_name": RUN_NAME,
            "run_profile": RUN_PROFILE,
            "slice": predictions[0]["slice"] if predictions else manifest_path.stem,
            "condition": predictions[0]["condition"] if predictions else "unknown",
            "num_beams": num_beams,
            "smoke_test_only": SMOKE_TEST_ONLY,
            "adapter": str(FINAL_ADAPTER_DIR) if model_label.endswith("lora") else None,
        },
        "samples": predictions,
    }
    out_json = ARTIFACTS_DIR / f"{RUN_NAME}_{model_label}_{manifest_path.stem}_beam{num_beams}.json"
    out_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"file": str(out_json), **payload["summary"]}


eval_manifests = build_eval_manifests()
all_summaries = []

for model_label, loader in [("hf_whisper_small_base", load_base_eval_model), ("hf_whisper_small_lora", load_lora_eval_model)]:
    eval_model = loader()
    for manifest_path in eval_manifests:
        for num_beams in DECODE_BEAMS:
            all_summaries.append(evaluate_model_on_manifest(eval_model, model_label, manifest_path, num_beams))
    del eval_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(all_summaries)
base_cols = ["slice", "num_beams", "files", "macro_wer", "macro_cer", "corpus_wer", "corpus_cer"]
base_df = summary_df[summary_df["model"] == "hf_whisper_small_base"][base_cols].rename(
    columns={
        "macro_wer": "base_macro_wer",
        "macro_cer": "base_macro_cer",
        "corpus_wer": "base_corpus_wer",
        "corpus_cer": "base_corpus_cer",
    }
)
lora_df = summary_df[summary_df["model"] == "hf_whisper_small_lora"][base_cols].rename(
    columns={
        "macro_wer": "lora_macro_wer",
        "macro_cer": "lora_macro_cer",
        "corpus_wer": "lora_corpus_wer",
        "corpus_cer": "lora_corpus_cer",
    }
)
comparison_df = base_df.merge(lora_df, on=["slice", "num_beams", "files"], how="outer")
comparison_df["delta_macro_wer"] = comparison_df["lora_macro_wer"] - comparison_df["base_macro_wer"]
comparison_df["delta_corpus_wer"] = comparison_df["lora_corpus_wer"] - comparison_df["base_corpus_wer"]
comparison_df = comparison_df.sort_values(["slice", "num_beams"])

summary_json_path = ARTIFACTS_DIR / f"{RUN_NAME}_eval_summary.json"
summary_md_path = ARTIFACTS_DIR / f"{RUN_NAME}_eval_summary.md"
comparison_md_path = ARTIFACTS_DIR / f"{RUN_NAME}_base_vs_lora_comparison.md"
summary_json_path.write_text(summary_df.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")
summary_md_path.write_text(summary_df.to_markdown(index=False), encoding="utf-8")
comparison_md_path.write_text(comparison_df.to_markdown(index=False), encoding="utf-8")
print("Per-run summaries:")
print(summary_df.to_markdown(index=False))
print()
print("Base vs LoRA comparison:")
print(comparison_df.to_markdown(index=False))
print("Wrote", summary_json_path)
print("Wrote", summary_md_path)
print("Wrote", comparison_md_path)


## Output Checklist

Kaggle keeps files under `/kaggle/working` as notebook outputs. The extracted dataset folders live under `/kaggle/temp/data` for the current run only.

Reusable dataset archives:

```text
/kaggle/working/saved_datasets/GV_Dev_5h.tar.gz
/kaggle/working/saved_datasets/GV_Train_100h.tar.gz
/kaggle/working/saved_datasets/gramvaani_saved_datasets.json
```

Experiment artifacts:

```text
/kaggle/working/results/whisper_small_lora_pilot/*_run_config.json
/kaggle/working/results/whisper_small_lora_pilot/*_train_split.csv
/kaggle/working/results/whisper_small_lora_pilot/*_internal_eval_split.csv
/kaggle/working/results/whisper_small_lora_pilot/*_excluded_rows.csv
/kaggle/working/results/whisper_small_lora_pilot/*_base_vs_lora_comparison.md
/kaggle/working/checkpoints/whisper-small-lora-gramvaani-*/final_adapter/
```

After the first successful run, create a Kaggle Dataset from `saved_datasets/` so future runs can attach the GramVaani tarballs instead of downloading them again.
